# 01. REINFORCE on a multi-armed bandit

A minimal example for the [post-training primer](https://www.norange.ai/post-training/2026/08/24/post-training-primer.html). Run the cells in order on a standard Colab CPU runtime; NumPy, PyTorch, and Matplotlib are included.

There are six arms. Choosing arm $a$ returns $r = \mu_a + \epsilon$, where $\epsilon \sim \mathcal N(0, 1)$. The policy only observes the reward of the arm it samples; the means define the simulator and are not passed to the optimizer.


## 1. Imports


In [ ]:
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt


## 2. Environment and policy

A softmax turns six trainable logits into action probabilities. Equal initial logits give a uniform policy. The seed makes this example reproducible in the same environment; try other seeds to see how noisy feedback changes learning.


In [ ]:
torch.manual_seed(42)

mean_scores = torch.tensor([0.1, 0.2, 1.0, 1.1, 0.5, 0.8])
policy_logits = torch.nn.Parameter(torch.ones(len(mean_scores)))
optimizer = optim.SGD([policy_logits], lr=0.05)

step_number = 4000
prob_history = []
reward_history = []


## 3. REINFORCE updates

Each episode consists of one action, so its return is the sampled reward. We minimize $L = -r\log\pi_\theta(a)$, which performs gradient ascent on expected reward. The sampled action and reward are treated as fixed when differentiating the log probability. This basic example uses no baseline.

Rerun the setup cell before repeating training to reset both the policy and optimizer.


In [ ]:
for _ in range(step_number):
    policy_probs = torch.softmax(policy_logits, dim=0)

    # Record the policy before this update.
    prob_history.append(policy_probs.detach().numpy().copy())

    # Sample an arm, then observe its noisy reward.
    action = torch.multinomial(policy_probs, num_samples=1).item()
    reward = mean_scores[action] + torch.randn(())
    reward_history.append(reward.item())

    # log_softmax computes the log probability stably.
    log_prob = torch.log_softmax(policy_logits, dim=0)[action]
    loss = -reward * log_prob

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


## 4. Plot learning

The left plot shows the action probabilities. The right plot smooths observed rewards with a 100-step moving average; the dashed line is the best arm's expected reward, not a ceiling on individual noisy rewards. This is one stochastic run: REINFORCE can concentrate on a suboptimal arm, especially when the best means are close. Try changing the seed, learning rate, or number of steps.


In [ ]:
prob_history = np.array(prob_history)
reward_history = np.array(reward_history)

window = 100
smoothed_rewards = np.convolve(
    reward_history,
    np.ones(window) / window,
    mode="valid",
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(len(mean_scores)):
    axes[0].plot(prob_history[:, i], label=f"Arm {i}")

axes[0].set_title("Policy probabilities")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Probability")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(
    np.arange(window - 1, step_number),
    smoothed_rewards,
    label="Moving average reward",
)
axes[1].axhline(
    mean_scores.max().item(),
    color="red",
    linestyle="--",
    label="Optimal expected reward",
)

axes[1].set_title("Reward progression")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Reward")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
